# Notebook completo — modelo preditivo de risco de defasagem

**Objetivo:** documentar o fluxo desde a **base bruta** até a **avaliação** do modelo que estima o risco de o estudante estar **em defasagem** (`defasagem` negativa na base harmonizada PEDE mesma regra que `defasagem < 0` no código).

**Roteiro:** (1) configuração e carga da base; (2) limpeza e unificação PEDE; (3) feature engineering; (4) treino/teste; (5) modelagem; (6) avaliação; (7) exportação opcional do `joblib`.

**Fonte da lógica (mantida nos `.py`):** limpeza em `src/pede_cleaning.py`; modelo em `src/pede_model.py` (também usado pelo Streamlit e por `scripts/train_model.py`). O notebook **importa** esses módulos em vez de duplicar código.

**Como executar:** execute as células **em ordem**. A primeira célula de código define `ROOT` e `sys.path`. Se pular a seção 2, o `df` vem só do parquet da seção 1; se rodar a seção 2, o `df` é recalculado a partir dos CSVs (`build_unified` em `src/pede_cleaning.py`).


## 1. Configuração e base de dados

- Definimos a raiz do projeto (`ROOT`), `sys.path` e carregamos `data_processed/pede_unificado.parquet`.
- Se o parquet ainda não existir, `ensure_parquet` em `src/pede_model.py` chama `build_unified` em `src/pede_cleaning.py` (mesmo fluxo do app).
- A **seção 2** mostra explicitamente a limpeza e a unificação a partir dos CSVs; depois dela, o `df` usado no restante do notebook é o recém-gerado.


In [ ]:
from pathlib import Path
import sys

# Raiz do projeto (pasta com src/ e data_processed/)
ROOT = Path.cwd()
if (ROOT / "src" / "pede_model.py").exists():
    pass
elif (ROOT.parent / "src" / "pede_model.py").exists():
    ROOT = ROOT.parent
else:
    ROOT = Path("..").resolve()

sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

import pandas as pd
from src.pede_model import ensure_parquet

pq = ensure_parquet(ROOT)
df = pd.read_parquet(pq)
print("Parquet:", pq)
print("Shape:", df.shape)
df.head()


## 2. Limpeza e unificação PEDE (2022–2024)

Esta seção reproduz o fluxo de limpeza/unificação usando **`src/pede_cleaning.py`**, que:

- remove colunas duplicadas do CSV (Destaque IPV; Ativo/Inativo);
- harmoniza nomes e tipos;
- empilha os três anos com chave `(ra, ano_cohorte)`;
- grava `data_processed/pede_unificado.parquet`.

**Requisito:** execute antes a **seção 1** (variável `ROOT` e `sys.path`). O `df` produzido aqui substitui o da célula anterior e alimenta as etapas de modelagem.

In [ ]:
from src.pede_cleaning import build_unified, cleaning_report

OUT_PARQUET = ROOT / "data_processed" / "pede_unificado.parquet"
df = build_unified(root=ROOT, save_parquet=OUT_PARQUET)
cleaning_report(df)

In [ ]:
df.head(10)

In [ ]:
dup = df.duplicated(subset=["ra", "ano_cohorte"])
assert not dup.any(), f"Duplicatas: {dup.sum()}"
print("OK: chave (ra, ano_cohorte) única.")

## 3. Engenharia de atributos (feature engineering)

Implementado em `build_xy` e em `make_pipeline` em **`src/pede_model.py`**:

| Etapa | Descrição |
|-------|-----------|
| **Alvo** | `y = 1` se `defasagem < 0`, senão `0` |
| **Features** | `NUMERIC_FEATURES` + `genero` — **não** usamos `ian` nem `defasagem` como preditores (evita vazamento do alvo) |
| **Categórica** | `genero` como string; vazios viram `"Desconhecido"` |
| **Filtro** | Mantemos linhas com **pelo menos um** indicador numérico não nulo |
| **Pré-processamento** | `ColumnTransformer`: mediana + `StandardScaler` nos numéricos; moda + `OneHotEncoder` em `genero` |


In [ ]:
import numpy as np
from src.pede_model import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_xy,
)

X, y = build_xy(df)
mask = X[NUMERIC_FEATURES].notna().any(axis=1)
X = X.loc[mask]
y = y[mask.values]

print("Número de features numéricas:", len(NUMERIC_FEATURES))
print("Features numéricas:", NUMERIC_FEATURES)
print("Features categóricas:", CATEGORICAL_FEATURES)
print("Taxa de positivos (defasagem menor que zero):", float(y.mean()))
X.describe(include="all").T.head(20)


## 4. Separação em treino e teste

Holdout **25%** para teste, `random_state=42`, amostragem **estratificada** por `y` — igual a `train_test_split` em `train_risk_model` em **`src/pede_model.py`**.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Treino:", X_train.shape[0], "| Teste:", X_test.shape[0])
print("Positivos — treino:", float(y_train.mean()), "| teste:", float(y_test.mean()))


## 5. Modelagem preditiva

`RandomForestClassifier` dentro de um **`sklearn.pipeline.Pipeline`** com pré-processamento (`make_pipeline` em **`src/pede_model.py`**): `n_estimators=400`, `max_depth=14`, `class_weight='balanced_subsample'`, etc.


In [ ]:
from src.pede_model import make_pipeline

pipe = make_pipeline()
pipe.fit(X_train, y_train)
print("Treinamento concluído.")
pipe


## 6. Avaliação dos resultados

- **ROC-AUC** no conjunto de teste (probabilidade da classe positiva).
- **Relatório de classificação** e **matriz de confusão** com limiar 0,5.
- Gráficos com **Plotly** (dependência do projeto em `requirements.txt`).


In [ ]:
from sklearn.metrics import auc, classification_report, confusion_matrix, roc_auc_score, roc_curve
import plotly.graph_objects as go

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

roc = float(roc_auc_score(y_test, proba))
print("ROC-AUC (teste):", roc)
print()
print(classification_report(y_test, pred, digits=3))

cm = confusion_matrix(y_test, pred)
print("Matriz de confusão [[TN FP] na primeira linha, [FN TP] na segunda]:")
print(cm)

fpr, tpr, _ = roc_curve(y_test, proba)
roc_auc_curve = auc(fpr, tpr)

fig_cm = go.Figure(
    data=go.Heatmap(
        z=cm[::-1],
        x=["Predito 0", "Predito 1"],
        y=["Real 1", "Real 0"],
        text=cm[::-1],
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=True,
    )
)
fig_cm.update_layout(
    title="Matriz de confusão (teste)",
    xaxis_title="Predito",
    yaxis_title="Real",
    width=520,
    height=420,
)
fig_cm.show()

fig_roc = go.Figure()
fig_roc.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"ROC (AUC = {roc_auc_curve:.3f})",
        fill="tozeroy",
        fillcolor="rgba(99,110,250,0.15)",
    )
)
fig_roc.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatório", line=dict(dash="dash", color="gray"))
)
fig_roc.update_layout(
    title="Curva ROC — conjunto de teste",
    xaxis_title="Taxa de falsos positivos",
    yaxis_title="Taxa de verdadeiros positivos",
    width=560,
    height=480,
    yaxis=dict(scaleanchor="x", scaleratio=1),
)
fig_roc.show()


### 6.1 Importância das variáveis (Random Forest)

Após o `OneHotEncoder`, os nomes das colunas transformadas vêm de `get_feature_names_out()` do `ColumnTransformer` no pipeline.


In [ ]:
import pandas as pd
import plotly.graph_objects as go

prep = pipe.named_steps["prep"]
clf = pipe.named_steps["clf"]
names = prep.get_feature_names_out()
imps = clf.feature_importances_
imp_df = pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False).head(25)

fig_imp = go.Figure(
    go.Bar(x=imp_df["importance"], y=imp_df["feature"], orientation="h", marker_color="#636EFA")
)
fig_imp.update_layout(
    title="Top 25 importâncias (espaço transformado pelo pipeline)",
    xaxis_title="Importância",
    yaxis_title="",
    height=700,
    margin=dict(l=160, r=24, t=60, b=48),
)
fig_imp.show()
imp_df
